# Phase 3 — LoRA Fine-Tuning — intent-classifier

This notebook:
1. Clones the `intent-classifier` repo from GitHub
2. Installs required Python packages (peft, trl, transformers, etc.)
3. Authenticates with Hugging Face
4. Checks GPU environment
5. Prepares train / val / test JSONL splits
6. Runs a 10-step smoke test to validate the pipeline
7. Trains LoRA adapters for selected models and configs
8. Evaluates on the validation set
9. Evaluates on the test set (locked — run deliberately)
10. Runs inference sanity checks on hand-picked examples
11. Generates analysis plots
12. Displays results inline
13. Downloads outputs as a zip archive

> **Runtime:** Set runtime type to **GPU → L4** (or A100) for training.  
> T4 will work for 1k dataset but may be slow for 10k.
>
> **Disk note:** Checkpoints for Config C (wide) can reach ~1–2 GB per model.  
> Mount Google Drive (optional cell below) to persist checkpoints across sessions.

## 1. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. Install dependencies

In [ ]:
%pip install -q \
    torch \
    transformers \
    accelerate \
    peft \
    trl \
    datasets \
    bitsandbytes \
    python-dotenv \
    sentencepiece \
    protobuf \
    matplotlib \
    numpy

import pkg_resources
for pkg in ["torch", "transformers", "peft", "trl", "datasets"]:
    try:
        v = pkg_resources.get_distribution(pkg).version
        print(f"  {pkg:<18} {v}")
    except Exception:
        print(f"  {pkg:<18} NOT FOUND")

## 3. Hugging Face authentication

Both Qwen2.5-0.5B-Instruct and Qwen3-0.6B are open-weight models that do **not** require a token.  
Store your token in Colab Secrets as **`HF_TOKEN`** (left panel → 🔑 icon) anyway to avoid rate-limiting on downloads.

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("HF_TOKEN secret not set — open-weight models will still work.")
except Exception:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in environment.")
    else:
        print("HF_TOKEN not found — open-weight models will still work.")

## 4. GPU environment check

In [ ]:
import torch, platform

print(f"Python     : {platform.python_version()}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    dev = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(dev)
    print(f"GPU        : {props.name}  (device {dev})")
    print(f"VRAM       : {props.total_memory / 1024**3:.1f} GB")
    print(f"BF16       : {torch.cuda.is_bf16_supported()}")
    cap = f"{props.major}.{props.minor}"
    print(f"Compute cap: {cap}")
else:
    print("No CUDA GPU detected — training will be very slow on CPU.")

## 5. Configuration

Edit the variables below to control which models and configs run and which dataset size to use.

| `DATASET_SIZE` | Train | Val | Test | Notes |
|---|---|---|---|---|
| `"1k"` | 800 | 100 | 100 | Fast calibration run |
| `"10k"` | 8 000 | 1 000 | 1 000 | Full-scale run |

In [ ]:
import sys

REPO_DIR  = "/content/intent-classifier"
SRC_DIR   = f"{REPO_DIR}/finetune_LoRA/src"
DATA_DIR  = f"{REPO_DIR}/dataset_full"
SPLIT_DIR = f"{REPO_DIR}/finetune_LoRA/data"
CKPT_DIR  = f"{REPO_DIR}/finetune_LoRA/checkpoints"
PLOT_DIR  = f"{REPO_DIR}/finetune_LoRA/analysis"

sys.path.insert(0, SRC_DIR)

# ── Dataset size toggle ────────────────────────────────────────────────────────
# "1k"  — first 10 files (1 000 examples): fast calibration, ~3-5 min/run on L4
# "10k" — all 100 files (10 000 examples): full scale,       ~30-50 min/run on L4
DATASET_SIZE = "1k"

# ── Models and configs to run ─────────────────────────────────────────────────
# Remove entries you want to skip.
MODELS_TO_RUN  = ["qwen2.5-0.5b", "qwen3-0.6b"]
CONFIGS_TO_RUN = ["A", "B", "C"]   # A=light, B=standard, C=wide

DEVICE = "auto"   # auto | cuda | cpu

print(f"Dataset size  : {DATASET_SIZE}")
print(f"Models        : {MODELS_TO_RUN}")
print(f"LoRA configs  : {CONFIGS_TO_RUN}")
print(f"Device        : {DEVICE}")
print(f"Total runs    : {len(MODELS_TO_RUN) * len(CONFIGS_TO_RUN)}")

## 6. (Optional) Mount Google Drive for persistent checkpoints

Skip this cell if you do not need checkpoints to survive a session restart.  
Colab provides ~100 GB of free Drive storage.

In [ ]:
# Uncomment the lines below to mount Drive and redirect checkpoints there.

# from google.colab import drive
# drive.mount("/content/drive")
# CKPT_DIR = "/content/drive/MyDrive/intent-classifier-lora/checkpoints"
# print(f"Checkpoints will be saved to: {CKPT_DIR}")

## 7. Prepare data splits

In [ ]:
import subprocess, sys, os

os.makedirs(SPLIT_DIR, exist_ok=True)

result = subprocess.run(
    [
        sys.executable,
        f"{SRC_DIR}/prepare_lora_data.py",
        "--dataset-size", DATASET_SIZE,
        "--data-dir",     DATA_DIR,
        "--out-dir",      SPLIT_DIR,
    ],
    check=True, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 8. Smoke test

Runs **10 training steps** on one model + config to validate the full pipeline  
(data loading → tokenisation → LoRA → SFTTrainer → callback → report write)  
without committing to a full training run.

Expected runtime: **< 2 minutes** on L4.

In [ ]:
import subprocess, sys, time

SMOKE_MODEL  = MODELS_TO_RUN[0]   # test with the first model in the list
SMOKE_CONFIG = "A"                # lightest config for fastest smoke test

print(f"Smoke test — model={SMOKE_MODEL}  config={SMOKE_CONFIG}  dataset={DATASET_SIZE}\n")
t0 = time.time()

result = subprocess.run(
    [
        sys.executable,
        f"{SRC_DIR}/lora_train.py",
        "--model",        SMOKE_MODEL,
        "--lora-config",  SMOKE_CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--ckpt-dir",     CKPT_DIR,
        "--device",       DEVICE,
        "--smoke-test",
    ],
    text=True,
)

elapsed = time.time() - t0
if result.returncode == 0:
    print(f"\n✅ Smoke test PASSED  ({elapsed:.0f}s)")
else:
    print(f"\n❌ Smoke test FAILED  (exit code {result.returncode})")

## 9. Train

Runs all (model × config) combinations sequentially.  
Estimated runtimes on **L4 GPU** for `DATASET_SIZE = "1k"`, 3 epochs:

| Config | Trainable params | Est. time / run |
|---|---|---|
| A (light)    | ~540K–800K   | ~3–5 min |
| B (standard) | ~2–3M        | ~5–8 min |
| C (wide)     | ~17M         | ~10–15 min |

Training stops early if `eval_loss` does not improve for 3 consecutive eval checkpoints.

In [ ]:
import subprocess, sys, time

TRAIN_SCRIPT = f"{SRC_DIR}/run_lora_experiments.py"

cmd = [
    sys.executable, TRAIN_SCRIPT,
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
]

print("Starting training...\n")
t0 = time.time()
result = subprocess.run(cmd, text=True)
elapsed = time.time() - t0

if result.returncode == 0:
    print(f"\n✅ Training complete  ({elapsed:.0f}s  /  {elapsed/60:.1f} min)")
else:
    print(f"\n❌ Training failed  (exit code {result.returncode})")

## 10. Validate (validation set)

Loads each trained adapter and evaluates on the validation split.  
Reports are written to `finetune_LoRA/reports_validation/`.

In [ ]:
import subprocess, sys

EVAL_SCRIPT = f"{SRC_DIR}/run_lora_experiments.py"

cmd = [
    sys.executable, EVAL_SCRIPT,
    "--models",        *MODELS_TO_RUN,
    "--configs",       *CONFIGS_TO_RUN,
    "--dataset-size",  DATASET_SIZE,
    "--device",        DEVICE,
    "--skip-training",   # only run val eval, skip re-training
]

result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print(f"\n❌ Validation failed  (exit code {result.returncode})")

## 11. Test evaluation (locked)

Run this cell deliberately once you are satisfied with validation results.  
For `DATASET_SIZE = "10k"`, two accuracy numbers are shown:
- **Anchor accuracy**: sample_0001 only (100 examples — same as the 1k test set)
- **Full test accuracy**: all 1 000 test examples

Reports are written to `finetune_LoRA/reports_test/`.

In [ ]:
import subprocess, sys

VALIDATE_SCRIPT = f"{SRC_DIR}/lora_validate.py"

for model_key in MODELS_TO_RUN:
    for cfg in CONFIGS_TO_RUN:
        print(f"\n── Test eval: {model_key} / config-{cfg} ──")
        result = subprocess.run(
            [
                sys.executable, VALIDATE_SCRIPT,
                "--model",        model_key,
                "--lora-config",  cfg,
                "--dataset-size", DATASET_SIZE,
                "--split",        "test",
                "--device",       DEVICE,
            ],
            text=True,
        )
        if result.returncode != 0:
            print(f"  ❌ FAILED  (exit code {result.returncode})")

## 12. Inference sanity check

Manually inspect model outputs on a few hand-picked examples.  
This catches output formatting regressions that a bulk accuracy number can hide:
- Does the model output exactly the tool name (no explanation, no extra tokens)?
- Does it correctly return `none` when no tool matches?
- Does it handle a rare tool correctly?

Edit `SANITY_MODEL` and `SANITY_CONFIG` to check any specific run.

In [ ]:
import json, sys, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

sys.path.insert(0, SRC_DIR)
from common import MODEL_REGISTRY, build_chat_messages, apply_chat_template_safe, extract_prediction  # type: ignore

SANITY_MODEL  = MODELS_TO_RUN[0]
SANITY_CONFIG = "B"

ckpt_path = Path(CKPT_DIR) / f"{SANITY_MODEL}_config_{SANITY_CONFIG}_{DATASET_SIZE}"
model_id  = MODEL_REGISTRY[SANITY_MODEL]
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading {SANITY_MODEL} + LoRA config-{SANITY_CONFIG} from {ckpt_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(ckpt_path))
base      = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map={"":device}
)
model     = PeftModel.from_pretrained(base, str(ckpt_path))
model.eval()
print("Model loaded.\n")

# Load the first 5 examples from test_anchor.jsonl as sanity examples
anchor_file = Path(SPLIT_DIR) / DATASET_SIZE / "test_anchor.jsonl"
sanity_examples = [
    json.loads(l) for l in anchor_file.read_text().splitlines() if l.strip()
][:5]

print(f"{'─'*70}")
print(f"  {'REQUEST':<35}  {'EXPECTED':<20}  {'PREDICTED':<20}  OK")
print(f"{'─'*70}")

for ex in sanity_examples:
    messages = build_chat_messages(ex, include_answer=False)
    text     = apply_chat_template_safe(tokenizer, messages, SANITY_MODEL, add_generation_prompt=True)
    inputs   = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=16, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    raw  = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = extract_prediction(raw)
    ok   = "✅" if pred == ex["answer"] else "❌"
    req  = ex["user_request"][:33]
    print(f"  {req:<35}  {ex['answer']:<20}  {pred:<20}  {ok}")

print(f"{'─'*70}")

# Cleanup
del model, base
import gc; gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()

## 13. Generate analysis plots

In [ ]:
import subprocess, sys, os

os.makedirs(PLOT_DIR, exist_ok=True)

PLOT_SCRIPT = f"{SRC_DIR}/plot_lora_results.py"

for split in ["val", "test"]:
    print(f"\nGenerating plots for split={split} ...")
    result = subprocess.run(
        [
            sys.executable, PLOT_SCRIPT,
            "--split",   split,
            "--out-dir", PLOT_DIR,
        ],
        text=True, capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)

## 14. Display results

In [ ]:
from IPython.display import display, Image  # type: ignore
from pathlib import Path

plot_files = sorted(Path(PLOT_DIR).glob("*.png"))

if not plot_files:
    print("No PNG files found in", PLOT_DIR)
else:
    for p in plot_files:
        print(f"\n── {p.name} ──")
        display(Image(filename=str(p)))

## 15. Download outputs (optional)

Zips and downloads:
- Training reports (`reports_training/`)
- Validation reports (`reports_validation/`)
- Test reports (`reports_test/`)
- Analysis plots (`analysis/`)

Checkpoints are excluded by default (can be several GB).  
Uncomment the `base_dirs` line to include them.

In [ ]:
import shutil, os
from pathlib import Path
from google.colab import files  # type: ignore

lora_dir  = Path(REPO_DIR) / "finetune_LoRA"
zip_base  = "/content/lora_results"
zip_path  = zip_base + ".zip"

# Collect files to include (exclude checkpoints by default)
include_dirs = [
    "reports_training",
    "reports_validation",
    "reports_test",
    "analysis",
    # "checkpoints",   # uncomment to include adapter weights (large!)
]

tmp_dir = Path("/content/_lora_export")
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir()

for d in include_dirs:
    src = lora_dir / d
    if src.exists():
        shutil.copytree(src, tmp_dir / d)
        n = sum(1 for _ in src.rglob("*") if _.is_file())
        print(f"  {d:<25} {n} files")

shutil.make_archive(zip_base, "zip", root_dir=str(tmp_dir), base_dir=".")
size_mb = os.path.getsize(zip_path) / 1024**2
print(f"\n  Archive: {zip_path}  ({size_mb:.1f} MB)")
files.download(zip_path)